In [ ]:
"""
11_train_window_sensitivity.py

Robustness check: does adding a second training year help?

07-10 use the baseline single-year setup (train on 2018, config.TRAIN_YEARS,
comparable to Cultrera & Bredart) and test on 2019. This notebook refits the
same 4 models on BOTH the baseline window (2018) and an EXTENDED training
window (2017+2018, config.TRAIN_YEARS_EXTENDED), and compares them side by
side, to see whether pooling a second training year actually improves
performance.

Both windows are fit fresh here rather than reading the 2018 numbers back
from logs/model_results.csv - reusing a cached row silently went stale in
practice whenever 07-10 were rerun (different config/features) after this
notebook had already logged its own comparison. The 2018 results are now
logged under the plain model name (the same key 07-10 use, so that row
stays in sync no matter which notebook ran last); the 2017-2018 results
keep their own "(train=2017-2018)" suffix so they never overwrite it.

Can be run independently of 07-10. 07-10 are still needed separately for
the richer per-model output (confusion matrix, feature importance, etc.)
used in the individual results sections.
"""


In [2]:
from utils.load_data_features import load_data_features

df = load_data_features()


In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from config import (
    FEATURES_BEHAVIORAL, FEATURES_FINANCIAL, RANDOM_STATE, TARGET,
    TRAIN_YEARS, TRAIN_YEARS_EXTENDED, WINSOR_COLUMNS,
)
from utils.model_results import save_model_results
from utils.print_section import print_section
from utils.time_based_split import time_based_split
from utils.winsorizer import Winsorizer

FEATURE_SETS = {
    "Financial": FEATURES_FINANCIAL,
    "Financial + Behavioral": FEATURES_FINANCIAL + FEATURES_BEHAVIORAL,
}

# (result label, which model, which feature set) - "result label" must
# match the plain model name 07-10 log under (MODEL_NAME in those
# notebooks), so the 2018 row logged below stays interchangeable with theirs.
MODEL_SPECS = [
    ("Logistic Regression", "Logistic Regression", "Financial"),
    ("Random Forest", "Random Forest", "Financial"),
    ("XGBoost Financial", "XGBoost", "Financial"),
    ("XGBoost Financial + Behavioral", "XGBoost", "Financial + Behavioral"),
]

TRAIN_WINDOWS = {
    "2018": TRAIN_YEARS,
    "2017-2018": TRAIN_YEARS_EXTENDED,
}


def make_pipeline(model_key, y_train):
    """Build the same pipeline as in 07-10, given which model to use.

    scale_pos_weight for XGBoost depends on y_train, so it has to be
    computed per training window, not reused across windows.
    """
    if model_key == "Logistic Regression":
        model = LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE,
        )
    elif model_key == "Random Forest":
        model = RandomForestClassifier(
            n_estimators=500, random_state=RANDOM_STATE,
            class_weight="balanced_subsample", n_jobs=-1,
        )
    elif model_key == "XGBoost":
        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
        model = XGBClassifier(
            n_estimators=500, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE, eval_metric="logloss",
        )
    else:
        raise ValueError(f"Unknown model_key: {model_key}")

    return Pipeline([
        ("winsorizer", Winsorizer(columns=WINSOR_COLUMNS)),
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])


# ------------------------------------------------------------------
# Fit every model fresh in BOTH training windows. The 2018 window is
# refit here too (not read back from logs/model_results.csv) so this
# notebook's comparison table can never go stale relative to whatever
# 07-10 happened to log last - a mismatch that showed up in practice
# when the CSV was updated by a later, differently-configured run of
# 07-10 than the one that had produced the previously cached numbers.
# ------------------------------------------------------------------
all_rows = []

for window_label, train_years in TRAIN_WINDOWS.items():
    print_section(f"Fitting training window {window_label}")

    for result_label, model_key, feature_set_key in MODEL_SPECS:
        features = FEATURE_SETS[feature_set_key]

        X_train, X_test, y_train, y_test = time_based_split(
            df, features, TARGET, train_years=train_years,
        )

        pipeline = make_pipeline(model_key, y_train)
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_prob),
            "pr_auc": average_precision_score(y_test, y_prob),
        }

        # 2018 keeps the plain model name (the same key 07-10 use), so this
        # refit keeps that log row in sync instead of leaving it to whichever
        # notebook happened to run last. 2017-2018 keeps its own suffixed
        # name so it never overwrites the 2018 baseline.
        log_name = (
            result_label if window_label == "2018"
            else f"{result_label} (train={window_label})"
        )
        save_model_results(log_name, metrics)

        all_rows.append({"model": result_label, "train_window": window_label, **metrics})

        print(
            f"{result_label:35s} "
            f"roc_auc={metrics['roc_auc']:.4f} "
            f"pr_auc={metrics['pr_auc']:.4f} "
            f"recall={metrics['recall']:.4f} "
            f"precision={metrics['precision']:.4f}"
        )

results_df = pd.DataFrame(all_rows)


In [ ]:
# ------------------------------------------------------------------
# Side-by-side comparison: baseline (2018) vs. extended (2017-2018),
# same model, same test year (2019).
# ------------------------------------------------------------------
print_section("Baseline (2018) vs. extended (2017-2018) - side by side")

comparison = results_df.pivot(
    index="model", columns="train_window", values=["roc_auc", "pr_auc", "recall", "precision"],
)

print(comparison.round(4))

# ------------------------------------------------------------------
# Training set size per window (row counts only, no fitting needed).
# ------------------------------------------------------------------
print_section("Training set size per window")

size_rows = []
for window_label, train_years in TRAIN_WINDOWS.items():
    _, _, y_train_window, _ = time_based_split(df, FEATURES_FINANCIAL, TARGET, train_years=train_years)
    size_rows.append({
        "train_window": window_label,
        "train_rows": len(y_train_window),
        "train_failures": int(y_train_window.sum()),
    })

print(pd.DataFrame(size_rows))
